# Etapa 1 - Setup do Ambiente

Nessa etapa o objetivo vai ser fazer a configuração do ambiente

In [4]:
# Instalação das dependências do projeto a partir do requirements.txt oficial,
# garantindo o mesmo ambiente de execução
!pip install -r requirements.txt --quiet --no-warn-conflicts

print("✅ Dependências instaladas com sucesso!")

✅ Dependências instaladas com sucesso!


In [ ]:
# Imports e configurações globais
import pandas as pd
import numpy as np
import os
import hashlib
from datetime import datetime, timezone
from colorama import Fore, Style, init
from tabulate import tabulate
import warnings
from google.cloud import bigquery
from dotenv import load_dotenv
import glob
from pathlib import Path
 
warnings.filterwarnings('ignore')

init(autoreset=True)

# Configuração de caminhos para simular um Data Lake local
BASE_PATH = "./data_lake"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
QUALITY_PATH = f"{BASE_PATH}/quality_reports"

for path in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
  os.makedirs(path, exist_ok=True)

# Timestamp de ingestão é gerado uma vez aqui e reutilizado em todas as etapas
INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

print(f"{Fore.GREEN} Ambiente configurado!")
print(f"{Fore.CYAN} Estrutura do Data Lake:")
print(f"    {BASE_PATH}/")
print(f"    ├── bronze/   -> Dados brutos (raw)")
print(f"    ├── silver/  -> Dados limpos e normalizados")
print(f"    ├── gold/    -> Dados agregados para consumo")
print(f"    └── quality_reports/ -> Relatórios de qualidade")
print(f"\n{Fore.YELLOW} Timestamp de ingestão: {INGESTION_TIMESTAMP}")

 Ambiente configurado!
 Estrutura do Data Lake:
    ./data_lake/
    ├── bronze/   -> Dados brutos (raw)
    ├── silver/  -> Dados limpos e normalizados
    ├── gold/    -> Dados agregados para consumo
    └── quality_reports/ -> Relatórios de qualidade

 Timestamp de ingestão: 20261101_181658


# Etapa 2 - Camada BRONZE: Ingestão de Dados

A camada Bronze armazena os dados exatamente como vieram da fonte — sem nenhuma transformação. Isso garante rastreabilidade e a possibilidade de reprocessamento.

In [59]:
# Optamos por consultar diretamente o BigQuery em vez de arquivos CSV 
# locais para garantir a governança, integridade e escalabilidade dos dados corporativos,
# evitando problemas de versionamento e limites de tamanho de arquivos estáticos.

In [6]:
# Carrega as configurações do arquivo externo .env
load_dotenv()

# Pega o caminho da credencial da variável de ambiente configurada no arquivo .env
cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if cred_path:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path

# Inicializa o cliente do BigQuery
client = bigquery.Client()

print(f"Conexão com o BigQuery configurada com sucesso!")

Conexão com o BigQuery configurada com sucesso!


In [7]:
def ingest_from_bigquery(table_id: str, entity_name: str, force_offline: bool = False) -> pd.DataFrame:
    """
    Ingere dados de uma tabela do BigQuery e salva na camada Bronze com histórico.
    Se force_offline=True ou se houver falha na API, utiliza o fallback local 
    emitindo um alerta explícito de auditoria.
    """
    output_file = f"{BRONZE_PATH}/{entity_name}_raw_{INGESTION_TIMESTAMP}.parquet"
    
    # Modo offline forçado (ótimo para testes locais sem gastar cota/rede)
    if force_offline:
        print(f"{Fore.YELLOW} [MODO OFFLINE ATIVADO] Pulando BigQuery para [{entity_name}].")
        return _carregar_fallback_recente(entity_name)

    print(f"{Fore.CYAN} Iniciando ingestão da entidade [{entity_name}] do BigQuery...")
    print(f"    Tabela fonte: {table_id}")

    try:
        # 1. Executa a consulta no BigQuery
        query = f"SELECT * FROM `{table_id}`"
        query_job = client.query(query)

        # EVIDÊNCIA DE NUVEM: Exibe o ID do Job executado diretamente nos servidores do GCP
        print(f"[Cloud GCP] Job ID gerado na nuvem: {query_job.job_id}")

        df = client.query(query).to_dataframe(create_bqstorage_client=True)

        # 2. Metadados e Hash de rastreabilidade
        df['_ingestion_timestamp'] = INGESTION_TIMESTAMP
        df['_source_table']        = table_id
        df['_source_system']       = 'basedosdados_bigquery'
        
        df['_record_hash'] = df.drop(
            columns=['_ingestion_timestamp', '_source_table', '_source_system'],
            errors='ignore'
        ).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

        # 3. Salva com histórico na Bronze
        df.to_parquet(output_file, index=False)

        print(f"{Fore.GREEN}    {len(df)} registros ingeridos do BigQuery com sucesso.")
        print(f"    Salvo com histórico em: {output_file}")
        return df

    except Exception as e:
        # ALERTA VERMELHO/AMARELO CLARO: O erro não fica escondido, ele é logado explicitamente
        print(f"\n{Fore.RED}========================================================")
        print(f"{Fore.RED} [ALERTA DE FALHA] Erro crítico ao conectar no BigQuery para [{entity_name}]:")
        print(f"{Fore.YELLOW} Detalhe do erro: {e}")
        print(f"{Fore.RED}========================================================\n")
        
        print(f"{Fore.YELLOW} [CONTINGÊNCIA] O pipeline acionou o fallback para evitar parada total.")
        return _carregar_fallback_recente(entity_name)

def _carregar_fallback_recente(entity_name: str) -> pd.DataFrame:
    """Função auxiliar para buscar o arquivo mais recente na Bronze com aviso explícito."""
    search_pattern = f"{BRONZE_PATH}/{entity_name}_raw_*.parquet"
    arquivos_existentes = glob.glob(search_pattern)
    
    if arquivos_existentes:
        latest_file = max(arquivos_existentes)
        print(f"{Fore.MAGENTA}    [AUDITORIA] Carregando cache local pré-existente: {latest_file}")
        df = pd.read_parquet(latest_file)
        return df
    else:
        print(f"{Fore.RED}    [ERRO FATAL] Nenhum cache local encontrado para {entity_name}. Impossível prosseguir.")
        raise FileNotFoundError(f"Nenhum dado local disponível para {entity_name}")

print(f"{Fore.GREEN} Função de ingestão configurada")

 Função de ingestão configurada


In [8]:
# ============================================================
# ETAPA 2.2 — INGESTÃO MULTI-ENTIDADE
# ============================================================
# Ingerimos as entidades do projeto de avaliação da alfabetização:
# - dicionario                → dicionário de dados / variáveis
# - municipio                → indicadores consolidados por município
# - uf                       → indicadores consolidados por UF
# - meta_alfabetizacao_brasil  → trajetória e metas nacionais
# - meta_alfabetizacao_uf      → trajetória e metas por UF
# - meta_alfabetizacao_municipio → trajetória e metas por município
# - alunos

print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)")
print(f"{Fore.YELLOW}{'='*60}\n")

# Dicionário com todas as entidades a ingerir do BigQuery (Base dos Dados)
entities_bigquery = {
    'municipio':                    'basedosdados.br_inep_avaliacao_alfabetizacao.municipio',
    'uf':                           'basedosdados.br_inep_avaliacao_alfabetizacao.uf',
    'meta_alfabetizacao_brasil':    'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil',
    'meta_alfabetizacao_uf':        'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf',
    'meta_alfabetizacao_municipio': 'basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio',
    'dicionario':                   'basedosdados.br_inep_avaliacao_alfabetizacao.dicionario',
    'alunos':                       'basedosdados.br_inep_avaliacao_alfabetizacao.alunos'  
}

bronze_dfs = {}

for entity, table_id in entities_bigquery.items():
    bronze_dfs[entity] = ingest_from_bigquery(table_id, entity)
    print()

print(f"{Fore.GREEN}{'='*60}")
print(f"{Fore.GREEN}  INGESTÃO CONCLUÍDA")
print(f"{Fore.GREEN}   Total de entidades: {len(bronze_dfs)}")
print(f"{Fore.GREEN}   Total de registros: {sum(len(df) for df in bronze_dfs.values())}")
print(f"{Fore.GREEN}{'='*60}")

  INICIANDO PIPELINE DE INGESTÃO (BIGQUERY)

 Iniciando ingestão da entidade [municipio] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.municipio
[Cloud GCP] Job ID gerado na nuvem: 47bd5d67-468d-40d4-96b6-b3da3d8fff5b
    23995 registros ingeridos do BigQuery com sucesso.
    Salvo com histórico em: ./data_lake/bronze/municipio_raw_20261101_181658.parquet

 Iniciando ingestão da entidade [uf] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.uf
[Cloud GCP] Job ID gerado na nuvem: f8d09a23-2e4b-4849-bd4d-5c561cbaa47b
    145 registros ingeridos do BigQuery com sucesso.
    Salvo com histórico em: ./data_lake/bronze/uf_raw_20261101_181658.parquet

 Iniciando ingestão da entidade [meta_alfabetizacao_brasil] do BigQuery...
    Tabela fonte: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
[Cloud GCP] Job ID gerado na nuvem: cbc0c3eb-5e31-4c37-be87-13b753da3657
    3 registros ingeridos do BigQuery com sucesso

In [9]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DOS DADOS BRUTOS
# ============================================================
# Sempre inspecionar os dados antes de processá-los!

print(f"{Fore.YELLOW} PRÉVIA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_dfs.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  Entidade: {entity.upper()}")
    print(f"   Shape (Linhas, Colunas): {df.shape} | Nulos totais: {df.isnull().sum().sum()}")
    print(f"   Tipos: { {col: str(dtype) for col, dtype in df.dtypes.items()} }")
    print()
    display(df.head(3))
    print()

 PRÉVIA DOS DADOS BRUTOS (Bronze)

──────────────────────────────────────────────────
  Entidade: MUNICIPIO
   Shape (Linhas, Colunas): (23995, 19) | Nulos totais: 103923
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,d1b6df4e80959ce1a5082a62c2f1db0c
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,4cda7a8d76f3afa7a02c2237a0868043
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,5b53ac3da3f4a8d4c8a16760b6a46b74



──────────────────────────────────────────────────
  Entidade: UF
   Shape (Linhas, Colunas): (145, 19) | Nulos totais: 630
   Tipos: {'ano': 'Int64', 'sigla_uf': 'object', 'serie': 'object', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'media_portugues': 'float64', 'proporcao_aluno_nivel_0': 'float64', 'proporcao_aluno_nivel_1': 'float64', 'proporcao_aluno_nivel_2': 'float64', 'proporcao_aluno_nivel_3': 'float64', 'proporcao_aluno_nivel_4': 'float64', 'proporcao_aluno_nivel_5': 'float64', 'proporcao_aluno_nivel_6': 'float64', 'proporcao_aluno_nivel_7': 'float64', 'proporcao_aluno_nivel_8': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,42ee6a943b5c1b9714c8e978c57f248b
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,eca51cb864fb5c8429b4b562aa79eee6
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.uf,basedosdados_bigquery,586978b39f93e4d089e198ec3945236a



──────────────────────────────────────────────────
  Entidade: META_ALFABETIZACAO_BRASIL
   Shape (Linhas, Colunas): (3, 15) | Nulos totais: 0
   Tipos: {'ano': 'Int64', 'rede': 'object', 'taxa_alfabetizacao': 'float64', 'meta_alfabetizacao_2024': 'float64', 'meta_alfabetizacao_2025': 'float64', 'meta_alfabetizacao_2026': 'float64', 'meta_alfabetizacao_2027': 'float64', 'meta_alfabetizacao_2028': 'float64', 'meta_alfabetizacao_2029': 'float64', 'meta_alfabetizacao_2030': 'float64', 'percentual_participacao': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,fc91a9077697cf8711b674e9889d2cab
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,14eac9e1c9a7e2bd9221c6cdd7647110
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.m...,basedosdados_bigquery,5730997b67cf1c69fa2b61018ffe34c0


... (4 outputs left out)


──────────────────────────────────────────────────
  Entidade: DICIONARIO
   Shape (Linhas, Colunas): (27, 9) | Nulos totais: 0
   Tipos: {'id_tabela': 'object', 'nome_coluna': 'object', 'chave': 'object', 'cobertura_temporal': 'object', 'valor': 'object', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,id_tabela,nome_coluna,chave,cobertura_temporal,valor,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,alunos,alfabetizado,0,(1),Não,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,ab2fea563eb9aaf844ce5653f4ee7715
1,alunos,alfabetizado,1,(1),Sim,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,ffafb21c590d647d55c00c816f5d0a43
2,alunos,preenchimento_caderno,0,(1),Prova não preenchida,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.d...,basedosdados_bigquery,9defe376b673a1f661852a24a066ea2b



──────────────────────────────────────────────────
  Entidade: ALUNOS
   Shape (Linhas, Colunas): (3867999, 16) | Nulos totais: 1026676
   Tipos: {'ano': 'Int64', 'id_municipio': 'object', 'id_escola': 'object', 'id_aluno': 'object', 'caderno': 'object', 'serie': 'object', 'rede': 'object', 'presenca': 'object', 'preenchimento_caderno': 'object', 'alfabetizado': 'object', 'proficiencia': 'float64', 'peso_aluno': 'float64', '_ingestion_timestamp': 'object', '_source_table': 'object', '_source_system': 'object', '_record_hash': 'object'}



,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingestion_timestamp,_source_table,_source_system,_record_hash
0,2023,1501808,60001610,15032484,1,2,3,0,0,0,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,ce62df4802fa63782e8669c83c4fd829
1,2023,3110509,60019111,31124547,1,2,3,0,0,0,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,c681a090384a450bde72ebdf00c790f5
2,2023,3170206,60021142,31146835,1,2,3,0,0,0,NaN,NaN,20261101_181658,basedosdados.br_inep_avaliacao_alfabetizacao.a...,basedosdados_bigquery,6b38ef9231297896a2abe7b0146ae91d


# Etapa 3 - Camada SILVER: Limpeza e Normalização

In [21]:
# ============================================================
# ETAPA 3.1 — TRANSFORMAÇÃO: MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: MUNICIPIO\n")

# Função auxiliar para registrar os checks da Silver em CSV de auditoria
def registrar_check_silver(regra, tabela, tipo, aprovado, valor, detalhe=""):
    report_dir = Path("./data_lake/quality_reports")
    report_dir.mkdir(parents=True, exist_ok=True)
    report_path = report_dir / "silver_checks.csv"
    
    nova_linha = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "etapa": "silver_municipio",
        "regra": regra,
        "tabela": tabela,
        "tipo": tipo,  # 'bloqueante' ou 'informativo'
        "aprovado": aprovado,  # True ou False
        "valor": str(valor),
        "detalhe": detalhe
    }
    
    df_log = pd.DataFrame([nova_linha])
    if report_path.exists():
        df_log.to_csv(report_path, mode='a', header=False, index=False)
    else:
        df_log.to_csv(report_path, mode='w', header=True, index=False)

# 1. Cópia do dataframe bronze
df_municipio_silver = bronze_dfs['municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_municipio_silver)
df_municipio_silver = df_municipio_silver.drop_duplicates()
linhas_depois = len(df_municipio_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_municipio_silver['id_municipio'] = df_municipio_silver['id_municipio'].astype(str)
df_municipio_silver['serie'] = df_municipio_silver['serie'].astype(str)
df_municipio_silver['rede'] = df_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_municipio_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_municipio_silver[col] = pd.to_numeric(df_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_municipio_silver.dtypes)

# 5. Checar se há nulos na chave principal (id_municipio)
n_nulos_id = int(df_municipio_silver['id_municipio'].isnull().sum())
aprovado_nulos = (n_nulos_id == 0)

registrar_check_silver(
    regra='id_municipio_nao_nulo',
    tabela='municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_nulos,
    valor=n_nulos_id,
    detalhe=f"{n_nulos_id} registros com id_municipio nulo."
)

if not aprovado_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade MUNICIPIO possui {n_nulos_id} registros com id_municipio nulo! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio (ano, id_municipio, serie, rede)
chaves_negocio = ['ano', 'id_municipio', 'serie', 'rede']
n_duplicadas = int(df_municipio_silver.duplicated(subset=chaves_negocio).sum())
aprovado_dup = (n_duplicadas == 0)

registrar_check_silver(
    regra='chave_negocio_unica',
    tabela='municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_dup,
    valor=n_duplicadas,
    detalhe=f"{n_duplicadas} linhas duplicadas para a chave de negócio."
)

if not aprovado_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade MUNICIPIO possui {n_duplicadas} linhas duplicadas para a mesma chave de negócio! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio (Taxa de Alfabetização entre 0 e 100)
# Filtramos os valores não nulos para verificar se há algum absurdo estatístico (ex: taxa 150%)
taxas_invalidas = int(((df_municipio_silver['taxa_alfabetizacao'] < 0) | (df_municipio_silver['taxa_alfabetizacao'] > 100)).sum())
aprovado_dom = (taxas_invalidas == 0)

registrar_check_silver(
    regra='dominio_taxa_alfabetizacao',
    tabela='municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_dom,
    valor=taxas_invalidas,
    detalhe=f"{taxas_invalidas} taxas fora do intervalo [0, 100]."
)

if not aprovado_dom:
    raise ValueError(f"[BLOQUEANTE] Encontradas {taxas_invalidas} taxas de alfabetização fora do intervalo [0, 100]. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Município) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_municipio_silver['_record_hash'] = df_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/municipio_silver.parquet"
df_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_municipio_silver)}")

  Processando entidade: MUNICIPIO

Duplicatas removidas: 0

Valores nulos por coluna:
ano                            0
id_municipio                   0
serie                          0
rede                           0
taxa_alfabetizacao             0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
_ingestion_timestamp           0
_source_table                  0
_source_system                 0
_record_hash                   0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64


In [22]:
# ============================================================
# ETAPA 3.2 — TRANSFORMAÇÃO: UF
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: UF\n")

# 1. Cópia do dataframe bronze
df_uf_silver = bronze_dfs['uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_uf_silver)
df_uf_silver = df_uf_silver.drop_duplicates()
linhas_depois = len(df_uf_silver)

print(f"Duplicatas removidas: {linhas_antes - linhas_depois}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_uf_silver['sigla_uf'] = df_uf_silver['sigla_uf'].astype(str)
df_uf_silver['serie'] = df_uf_silver['serie'].astype(str)
df_uf_silver['rede'] = df_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao', 'media_portugues'] + [col for col in df_uf_silver.columns if 'proporcao_aluno_nivel' in col]
for col in colunas_numericas:
    df_uf_silver[col] = pd.to_numeric(df_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_uf_silver.dtypes)

# 5. Checar se há nulos na chave principal (sigla_uf)
n_nulos_sigla = int(df_uf_silver['sigla_uf'].isnull().sum())
aprovado_sigla = (n_nulos_sigla == 0)

registrar_check_silver(
    regra='sigla_uf_nao_nula',
    tabela='uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_sigla,
    valor=n_nulos_sigla,
    detalhe=f"{n_nulos_sigla} registros com sigla_uf nula."
)

if not aprovado_sigla:
    raise ValueError(f"[BLOQUEANTE] A entidade UF possui {n_nulos_sigla} registros com sigla_uf nula! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio (ano, sigla_uf, serie, rede)
chaves_negocio_uf = ['ano', 'sigla_uf', 'serie', 'rede']
n_duplicadas_uf = int(df_uf_silver.duplicated(subset=chaves_negocio_uf).sum())
aprovado_dup_uf = (n_duplicadas_uf == 0)

registrar_check_silver(
    regra='chave_negocio_uf_unica',
    tabela='uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_dup_uf,
    valor=n_duplicadas_uf,
    detalhe=f"{n_duplicadas_uf} linhas duplicadas para a chave de negócio da UF."
)

if not aprovado_dup_uf:
    raise ValueError(f"[BLOQUEANTE] A entidade UF possui {n_duplicadas_uf} linhas duplicadas para a mesma chave de negócio! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio (Taxa de Alfabetização entre 0 e 100)
taxas_invalidas_uf = int(((df_uf_silver['taxa_alfabetizacao'] < 0) | (df_uf_silver['taxa_alfabetizacao'] > 100)).sum())
aprovado_dom_uf = (taxas_invalidas_uf == 0)

registrar_check_silver(
    regra='dominio_taxa_alfabetizacao_uf',
    tabela='uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_dom_uf,
    valor=taxas_invalidas_uf,
    detalhe=f"{taxas_invalidas_uf} taxas fora do intervalo [0, 100] na UF."
)

if not aprovado_dom_uf:
    raise ValueError(f"[BLOQUEANTE] Encontradas {taxas_invalidas_uf} taxas de alfabetização fora do intervalo [0, 100] na entidade UF. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (UF) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_uf_silver['_record_hash'] = df_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/uf_silver.parquet"
df_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_uf_silver)}")

  Processando entidade: UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                         0
sigla_uf                    0
serie                       0
rede                        0
taxa_alfabetizacao          0
media_portugues             0
proporcao_aluno_nivel_0    70
proporcao_aluno_nivel_1    70
proporcao_aluno_nivel_2    70
proporcao_aluno_nivel_3    70
proporcao_aluno_nivel_4    70
proporcao_aluno_nivel_5    70
proporcao_aluno_nivel_6    70
proporcao_aluno_nivel_7    70
proporcao_aluno_nivel_8    70
_ingestion_timestamp        0
_source_table               0
_source_system              0
_record_hash                0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                    object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    fl

In [23]:
# ============================================================
# ETAPA 3.3 — TRANSFORMAÇÃO: META_ALFABETIZACAO_BRASIL
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: META_ALFABETIZACAO_BRASIL\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_brasil_silver = bronze_dfs['meta_alfabetizacao_brasil'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_brasil_silver)
df_meta_alfabetizacao_brasil_silver = df_meta_alfabetizacao_brasil_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_brasil_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_brasil_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_brasil_silver['rede'] = df_meta_alfabetizacao_brasil_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_brasil_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_brasil_silver[col] = pd.to_numeric(df_meta_alfabetizacao_brasil_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_brasil_silver.dtypes)

# 5. Checar se há nulos nas chaves principais (ano e rede)
n_nulos_chaves_brasil = int(df_meta_alfabetizacao_brasil_silver[['ano', 'rede']].isnull().sum().sum())
aprovado_brasil_nulos = (n_nulos_chaves_brasil == 0)

registrar_check_silver(
    regra='chaves_brasil_nao_nulas',
    tabela='meta_alfabetizacao_brasil_silver',
    tipo='bloqueante',
    aprovado=aprovado_brasil_nulos,
    valor=n_nulos_chaves_brasil,
    detalhe=f"{n_nulos_chaves_brasil} valores nulos nas chaves principais (ano/rede)."
)

if not aprovado_brasil_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_BRASIL possui valores nulos nas chaves principais (ano/rede)! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio (ano, rede)
chaves_negocio_brasil = ['ano', 'rede']
n_duplicadas_brasil = int(df_meta_alfabetizacao_brasil_silver.duplicated(subset=chaves_negocio_brasil).sum())
aprovado_brasil_dup = (n_duplicadas_brasil == 0)

registrar_check_silver(
    regra='chave_negocio_brasil_unica',
    tabela='meta_alfabetizacao_brasil_silver',
    tipo='bloqueante',
    aprovado=aprovado_brasil_dup,
    valor=n_duplicadas_brasil,
    detalhe=f"{n_duplicadas_brasil} linhas duplicadas para a chave de negócio (ano/rede)."
)

if not aprovado_brasil_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_BRASIL possui {n_duplicadas_brasil} linhas duplicadas para a mesma chave de negócio! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio (Taxa de Alfabetização entre 0 e 100)
taxas_invalidas_brasil = int(((df_meta_alfabetizacao_brasil_silver['taxa_alfabetizacao'] < 0) | (df_meta_alfabetizacao_brasil_silver['taxa_alfabetizacao'] > 100)).sum())
aprovado_brasil_dom = (taxas_invalidas_brasil == 0)

registrar_check_silver(
    regra='dominio_taxa_alfabetizacao_brasil',
    tabela='meta_alfabetizacao_brasil_silver',
    tipo='bloqueante',
    aprovado=aprovado_brasil_dom,
    valor=taxas_invalidas_brasil,
    detalhe=f"{taxas_invalidas_brasil} taxas fora do intervalo [0, 100] na meta do Brasil."
)

if not aprovado_brasil_dom:
    raise ValueError(f"[BLOQUEANTE] Encontradas {taxas_invalidas_brasil} taxas fora do intervalo [0, 100] na entidade META_ALFABETIZACAO_BRASIL. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Meta Brasil) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_brasil_silver['_record_hash'] = df_meta_alfabetizacao_brasil_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_brasil_silver.parquet"
df_meta_alfabetizacao_brasil_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_BRASIL processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_brasil_silver)}")

  Processando entidade: META_ALFABETIZACAO_BRASIL

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
rede                       0
taxa_alfabetizacao         0
meta_alfabetizacao_2024    0
meta_alfabetizacao_2025    0
meta_alfabetizacao_2026    0
meta_alfabetizacao_2027    0
meta_alfabetizacao_2028    0
meta_alfabetizacao_2029    0
meta_alfabetizacao_2030    0
percentual_participacao    0
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_participacao    float64
_ingestion_timestamp        object
_

In [24]:
# ============================================================
# ETAPA 3.4 — TRANSFORMAÇÃO: META_ALFABETIZACAO_UF
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: META_ALFABETIZACAO_UF\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_uf_silver = bronze_dfs['meta_alfabetizacao_uf'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_uf_silver)
df_meta_alfabetizacao_uf_silver = df_meta_alfabetizacao_uf_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_uf_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_uf_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_uf_silver['sigla_uf'] = df_meta_alfabetizacao_uf_silver['sigla_uf'].astype(str)
df_meta_alfabetizacao_uf_silver['rede'] = df_meta_alfabetizacao_uf_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_uf_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_uf_silver[col] = pd.to_numeric(df_meta_alfabetizacao_uf_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_uf_silver.dtypes)

# 5. Checar se há nulos nas chaves principais (ano, sigla_uf, rede)
n_nulos_chaves_meta_uf = int(df_meta_alfabetizacao_uf_silver[['ano', 'sigla_uf', 'rede']].isnull().sum().sum())
aprovado_meta_uf_nulos = (n_nulos_chaves_meta_uf == 0)

registrar_check_silver(
    regra='chaves_meta_uf_nao_nulas',
    tabela='meta_alfabetizacao_uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_uf_nulos,
    valor=n_nulos_chaves_meta_uf,
    detalhe=f"{n_nulos_chaves_meta_uf} valores nulos nas chaves principais (ano/sigla_uf/rede)."
)

if not aprovado_meta_uf_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_UF possui valores nulos nas chaves principais (ano/sigla_uf/rede)! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio (ano, sigla_uf, rede)
chaves_negocio_meta_uf = ['ano', 'sigla_uf', 'rede']
n_duplicadas_meta_uf = int(df_meta_alfabetizacao_uf_silver.duplicated(subset=chaves_negocio_meta_uf).sum())
aprovado_meta_uf_dup = (n_duplicadas_meta_uf == 0)

registrar_check_silver(
    regra='chave_negocio_meta_uf_unica',
    tabela='meta_alfabetizacao_uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_uf_dup,
    valor=n_duplicadas_meta_uf,
    detalhe=f"{n_duplicadas_meta_uf} linhas duplicadas para a chave de negócio (ano/sigla_uf/rede)."
)

if not aprovado_meta_uf_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_UF possui {n_duplicadas_meta_uf} linhas duplicadas para a mesma chave de negócio! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio (Taxa de Alfabetização entre 0 e 100)
taxas_invalidas_meta_uf = int(((df_meta_alfabetizacao_uf_silver['taxa_alfabetizacao'] < 0) | (df_meta_alfabetizacao_uf_silver['taxa_alfabetizacao'] > 100)).sum())
aprovado_meta_uf_dom = (taxas_invalidas_meta_uf == 0)

registrar_check_silver(
    regra='dominio_taxa_alfabetizacao_meta_uf',
    tabela='meta_alfabetizacao_uf_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_uf_dom,
    valor=taxas_invalidas_meta_uf,
    detalhe=f"{taxas_invalidas_meta_uf} taxas fora do intervalo [0, 100] na meta da UF."
)

if not aprovado_meta_uf_dom:
    raise ValueError(f"[BLOQUEANTE] Encontradas {taxas_invalidas_meta_uf} taxas fora do intervalo [0, 100] na entidade META_ALFABETIZACAO_UF. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Meta UF) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_uf_silver['_record_hash'] = df_meta_alfabetizacao_uf_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet"
df_meta_alfabetizacao_uf_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_UF processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_uf_silver)}")

  Processando entidade: META_ALFABETIZACAO_UF

Duplicatas removidas: 0

Valores nulos por coluna:
ano                        0
sigla_uf                   0
rede                       0
taxa_alfabetizacao         4
meta_alfabetizacao_2024    9
meta_alfabetizacao_2025    3
meta_alfabetizacao_2026    2
meta_alfabetizacao_2027    2
meta_alfabetizacao_2028    2
meta_alfabetizacao_2029    2
meta_alfabetizacao_2030    2
percentual_participacao    4
_ingestion_timestamp       0
_source_table              0
_source_system             0
_record_hash               0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
sigla_uf                    object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_

In [25]:
# ============================================================
# ETAPA 3.5 — TRANSFORMAÇÃO: META_ALFABETIZACAO_MUNICIPIO
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: META_ALFABETIZACAO_MUNICIPIO\n")

# 1. Cópia do dataframe bronze
df_meta_alfabetizacao_municipio_silver = bronze_dfs['meta_alfabetizacao_municipio'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_meta_alfabetizacao_municipio_silver)
df_meta_alfabetizacao_municipio_silver = df_meta_alfabetizacao_municipio_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_meta_alfabetizacao_municipio_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_meta_alfabetizacao_municipio_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
df_meta_alfabetizacao_municipio_silver['id_municipio'] = df_meta_alfabetizacao_municipio_silver['id_municipio'].astype(str)
df_meta_alfabetizacao_municipio_silver['rede'] = df_meta_alfabetizacao_municipio_silver['rede'].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['taxa_alfabetizacao'] + [col for col in df_meta_alfabetizacao_municipio_silver.columns if 'meta_alfabetizacao_' in col]
for col in colunas_numericas:
    df_meta_alfabetizacao_municipio_silver[col] = pd.to_numeric(df_meta_alfabetizacao_municipio_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_meta_alfabetizacao_municipio_silver.dtypes)

# 5. Checar se há nulos nas chaves principais (ano, id_municipio, rede)
n_nulos_chaves_meta_mun = int(df_meta_alfabetizacao_municipio_silver[['ano', 'id_municipio', 'rede']].isnull().sum().sum())
aprovado_meta_mun_nulos = (n_nulos_chaves_meta_mun == 0)

registrar_check_silver(
    regra='chaves_meta_municipio_nao_nulas',
    tabela='meta_alfabetizacao_municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_mun_nulos,
    valor=n_nulos_chaves_meta_mun,
    detalhe=f"{n_nulos_chaves_meta_mun} valores nulos nas chaves principais (ano/id_municipio/rede)."
)

if not aprovado_meta_mun_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_MUNICIPIO possui valores nulos nas chaves principais (ano/id_municipio/rede)! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio (ano, id_municipio, rede)
chaves_negocio_meta_mun = ['ano', 'id_municipio', 'rede']
n_duplicadas_meta_mun = int(df_meta_alfabetizacao_municipio_silver.duplicated(subset=chaves_negocio_meta_mun).sum())
aprovado_meta_mun_dup = (n_duplicadas_meta_mun == 0)

registrar_check_silver(
    regra='chave_negocio_meta_municipio_unica',
    tabela='meta_alfabetizacao_municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_mun_dup,
    valor=n_duplicadas_meta_mun,
    detalhe=f"{n_duplicadas_meta_mun} linhas duplicadas para a chave de negócio (ano/id_municipio/rede)."
)

if not aprovado_meta_mun_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade META_ALFABETIZACAO_MUNICIPIO possui {n_duplicadas_meta_mun} linhas duplicadas para a mesma chave de negócio! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio (Taxa de Alfabetização entre 0 e 100, se preenchida)
taxas_invalidas_meta_mun = int(((df_meta_alfabetizacao_municipio_silver['taxa_alfabetizacao'] < 0) | (df_meta_alfabetizacao_municipio_silver['taxa_alfabetizacao'] > 100)).sum())
aprovado_meta_mun_dom = (taxas_invalidas_meta_mun == 0)

registrar_check_silver(
    regra='dominio_taxa_alfabetizacao_meta_municipio',
    tabela='meta_alfabetizacao_municipio_silver',
    tipo='bloqueante',
    aprovado=aprovado_meta_mun_dom,
    valor=taxas_invalidas_meta_mun,
    detalhe=f"{taxas_invalidas_meta_mun} taxas fora do intervalo [0, 100] na meta do município."
)

if not aprovado_meta_mun_dom:
    raise ValueError(f"[BLOQUEANTE] Encontradas {taxas_invalidas_meta_mun} taxas fora do intervalo [0, 100] na entidade META_ALFABETIZACAO_MUNICIPIO. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Meta Município) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_meta_alfabetizacao_municipio_silver['_record_hash'] = df_meta_alfabetizacao_municipio_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet"
df_meta_alfabetizacao_municipio_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade META_ALFABETIZACAO_MUNICIPIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_meta_alfabetizacao_municipio_silver)}")

  Processando entidade: META_ALFABETIZACAO_MUNICIPIO

Duplicatas removidas: 0

Valores nulos por coluna:
ano                          0
id_municipio                 0
rede                         0
taxa_alfabetizacao         120
meta_alfabetizacao_2024    240
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
_ingestion_timestamp         0
_source_table                0
_source_system               0
_record_hash                 0
dtype: int64

Tipos de dados atualizados:
ano                          Int64
id_municipio                object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabe

In [26]:
# ============================================================
# ETAPA 3.6 — TRANSFORMAÇÃO: DICIONARIO
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: DICIONARIO\n")

# 1. Cópia do dataframe bronze
df_dicionario_silver = bronze_dfs['dicionario'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_dicionario_silver)
df_dicionario_silver = df_dicionario_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_dicionario_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_dicionario_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_tabela', 'nome_coluna', 'chave', 'cobertura_temporal', 'valor']
for col in colunas_texto:
    if col in df_dicionario_silver.columns:
        df_dicionario_silver[col] = df_dicionario_silver[col].astype(str)

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_dicionario_silver.dtypes)

# 5. Checar se há nulos nas chaves de mapeamento (id_tabela, nome_coluna, chave, valor)
chaves_dic = ['id_tabela', 'nome_coluna', 'chave', 'valor']
n_nulos_dic = int(df_dicionario_silver[chaves_dic].isnull().sum().sum())
aprovado_dic_nulos = (n_nulos_dic == 0)

registrar_check_silver(
    regra='chaves_dicionario_nao_nulas',
    tabela='dicionario_silver',
    tipo='bloqueante',
    aprovado=aprovado_dic_nulos,
    valor=n_nulos_dic,
    detalhe=f"{n_nulos_dic} valores nulos em colunas obrigatórias do dicionário."
)

if not aprovado_dic_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade DICIONARIO possui valores nulos em colunas obrigatórias! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave primária do dicionário (id_tabela, nome_coluna, chave)
chave_primaria_dic = ['id_tabela', 'nome_coluna', 'chave']
n_duplicadas_dic = int(df_dicionario_silver.duplicated(subset=chave_primaria_dic).sum())
aprovado_dic_dup = (n_duplicadas_dic == 0)

registrar_check_silver(
    regra='chave_primaria_dicionario_unica',
    tabela='dicionario_silver',
    tipo='bloqueante',
    aprovado=aprovado_dic_dup,
    valor=n_duplicadas_dic,
    detalhe=f"{n_duplicadas_dic} mapeamentos duplicados para id_tabela/nome_coluna/chave."
)

if not aprovado_dic_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade DICIONARIO possui {n_duplicadas_dic} mapeamentos duplicados para a mesma combinação id_tabela/nome_coluna/chave! Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Dicionário) aprovadas com sucesso!{Style.RESET_ALL}")

# 7. Atualização do hash de registro e salvamento na Silver
df_dicionario_silver['_record_hash'] = df_dicionario_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/dicionario_silver.parquet"
df_dicionario_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade DICIONARIO processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_dicionario_silver)}")

  Processando entidade: DICIONARIO

Duplicatas removidas: 0

Valores nulos por coluna:
id_tabela               0
nome_coluna             0
chave                   0
cobertura_temporal      0
valor                   0
_ingestion_timestamp    0
_source_table           0
_source_system          0
_record_hash            0
dtype: int64

Tipos de dados atualizados:
id_tabela               object
nome_coluna             object
chave                   object
cobertura_temporal      object
valor                   object
_ingestion_timestamp    object
_source_table           object
_source_system          object
_record_hash            object
dtype: object
Validações de qualidade da Silver (Dicionário) aprovadas com sucesso!
 Entidade DICIONARIO processada com sucesso na camada Silver!
 Salvo em: ./data_lake/silver/dicionario_silver.parquet
 Total de registros: 27


In [27]:
# ============================================================
# ETAPA 3.7 — TRANSFORMAÇÃO: ALUNOS
# ============================================================
print(f"{Fore.YELLOW}  Processando entidade: ALUNOS\n")

# 1. Cópia do dataframe bronze
df_alunos_silver = bronze_dfs['alunos'].copy()

# 2. Remoção de duplicatas exatas, se houver
linhas_antes = len(df_alunos_silver)
df_alunos_silver = df_alunos_silver.drop_duplicates()
print(f"Duplicatas removidas: {linhas_antes - len(df_alunos_silver)}")

# 3. Verificação de valores nulos iniciais nas colunas principais
print("\nValores nulos por coluna:")
print(df_alunos_silver.isnull().sum())

# 4. Padronização de tipos de dados
# Garantindo que colunas categóricas/identificadores sejam string e métricas sejam numéricas
colunas_texto = ['id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado']
for col in colunas_texto:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = df_alunos_silver[col].astype(str)

# Convertendo colunas numéricas para float (caso alguma venha como object)
colunas_numericas = ['proficiencia', 'peso_aluno']
for col in colunas_numericas:
    if col in df_alunos_silver.columns:
        df_alunos_silver[col] = pd.to_numeric(df_alunos_silver[col], errors='coerce')

# Verificando os tipos após a conversão
print("\nTipos de dados atualizados:")
print(df_alunos_silver.dtypes)

# 5. Checar se há nulos nas chaves principais de identificação
chaves_alunos = ['ano', 'id_municipio', 'id_escola', 'id_aluno']
n_nulos_alunos = int(df_alunos_silver[chaves_alunos].isnull().sum().sum())
aprovado_alunos_nulos = (n_nulos_alunos == 0)

registrar_check_silver(
    regra='chaves_alunos_nao_nulas',
    tabela='alunos_silver',
    tipo='bloqueante',
    aprovado=aprovado_alunos_nulos,
    valor=n_nulos_alunos,
    detalhe=f"{n_nulos_alunos} valores nulos nas chaves de identificação principal dos alunos."
)

if not aprovado_alunos_nulos:
    raise ValueError(f"[BLOQUEANTE] A entidade ALUNOS possui valores nulos nas chaves de identificação principal! Verifique o relatório em quality_reports/silver_checks.csv.")

# 6. Checar duplicidades na chave de negócio do aluno 
# Como um aluno pode mudar de ano, a chave composta por ano + id_aluno deve ser única
chaves_negocio_aluno = ['ano', 'id_aluno']
n_duplicadas_aluno = int(df_alunos_silver.duplicated(subset=chaves_negocio_aluno).sum())
aprovado_alunos_dup = (n_duplicadas_aluno == 0)

registrar_check_silver(
    regra='chave_negocio_aluno_unica',
    tabela='alunos_silver',
    tipo='bloqueante',
    aprovado=aprovado_alunos_dup,
    valor=n_duplicadas_aluno,
    detalhe=f"{n_duplicadas_aluno} registos duplicados para o mesmo aluno no mesmo ano."
)

if not aprovado_alunos_dup:
    raise ValueError(f"[BLOQUEANTE] A entidade ALUNOS possui {n_duplicadas_aluno} registos duplicados para o mesmo aluno no mesmo ano! Verifique o relatório em quality_reports/silver_checks.csv.")

# 7. Validação de domínio prudente (Peso do aluno não pode ser negativo, quando preenchido)
pesos_invalidos = int((df_alunos_silver['peso_aluno'] < 0).sum())
aprovado_alunos_peso = (pesos_invalidos == 0)

registrar_check_silver(
    regra='dominio_peso_aluno_nao_negativo',
    tabela='alunos_silver',
    tipo='bloqueante',
    aprovado=aprovado_alunos_peso,
    valor=pesos_invalidos,
    detalhe=f"{pesos_invalidos} registos com peso de aluno negativo."
)

if not aprovado_alunos_peso:
    raise ValueError(f"[BLOQUEANTE] Encontrados {pesos_invalidos} registos com peso de aluno negativo. Verifique o relatório em quality_reports/silver_checks.csv.")

print(f"{Fore.GREEN}Validações de qualidade da Silver (Alunos) aprovadas com sucesso!{Style.RESET_ALL}")

# 8. Atualização do hash de registro e salvamento na Silver
df_alunos_silver['_record_hash'] = df_alunos_silver.drop(
    columns=['_ingestion_timestamp', '_source_table', '_source_system', '_record_hash'],
    errors='ignore'
).apply(lambda row: hashlib.md5(str(row.values).encode()).hexdigest(), axis=1)

# Caminho e salvamento do arquivo Parquet da camada Silver
output_file_silver = f"{SILVER_PATH}/alunos_silver.parquet"
df_alunos_silver.to_parquet(output_file_silver, index=False)

print(f"{Fore.GREEN} Entidade ALUNOS processada com sucesso na camada Silver!")
print(f" Salvo em: {output_file_silver}")
print(f" Total de registros: {len(df_alunos_silver)}")

  Processando entidade: ALUNOS

Duplicatas removidas: 0

Valores nulos por coluna:
ano                           0
id_municipio                  0
id_escola                     0
id_aluno                      0
caderno                       0
serie                         0
rede                          0
presenca                      0
preenchimento_caderno         0
alfabetizado                  0
proficiencia             513338
peso_aluno               513338
_ingestion_timestamp          0
_source_table                 0
_source_system                0
_record_hash                  0
dtype: int64

Tipos de dados atualizados:
ano                        Int64
id_municipio              object
id_escola                 object
id_aluno                  object
caderno                   object
serie                     object
rede                      object
presenca                  object
preenchimento_caderno     object
alfabetizado              object
proficiencia             float64


# Etapa 4 - Camada GOLD: Agregações e Métricas de Negócio

In [71]:
# ============================================================
# ETAPA 4.1 — CONSOLIDAÇÃO MUNICIPIO E METAS
# ============================================================
print(f"{Fore.YELLOW}Processando consolidação Gold: Município e Metas\n")

# Dicionário para armazenar os dataframes da camada Gold
gold_dfs = {}

# 1. Carregar os dados limpos da Silver
df_municipio_silver = pd.read_parquet(f"{SILVER_PATH}/municipio_silver.parquet")
df_meta_municipio_silver = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_municipio_silver.parquet")
df_dicionario_silver = pd.read_parquet(f"{SILVER_PATH}/dicionario_silver.parquet")

# 2. Criar o mapeamento usando a tabela de dicionário
df_map_rede = df_dicionario_silver[
    (df_dicionario_silver['id_tabela'] == 'municipio') & 
    (df_dicionario_silver['nome_coluna'] == 'rede')
][['chave', 'valor']].set_index('chave')['valor'].to_dict()

df_municipio_silver['rede_descricao'] = df_municipio_silver['rede'].map(df_map_rede)

# VERIFICAÇÃO / AUDITORIA: Conta se há valores que não estavam no dicionário
nulos_pos_map = df_municipio_silver['rede_descricao'].isnull().sum()
if nulos_pos_map > 0:
    print(f"{Fore.YELLOW}[ALERTA DE AUDITORIA] Encontrados {nulos_pos_map} registros com códigos fora do dicionário na coluna 'rede'. Atribuindo 'Desconhecido'.")
    df_municipio_silver['rede_descricao'] = df_municipio_silver['rede_descricao'].fillna('Desconhecido')
else:
    print(f"{Fore.GREEN}[AUDITORIA] Todos os códigos de 'rede' foram mapeados com sucesso pelo dicionário.")

# Na tabela de metas, a coluna de rede já está descritiva
df_meta_municipio_silver['rede_descricao'] = df_meta_municipio_silver['rede']

# 3. Realizando o inner join utilizando as descrições padronizadas
total_linhas_originais_municipio = len(df_municipio_silver)

df_gold_municipio = pd.merge(
    df_municipio_silver,
    df_meta_municipio_silver,
    left_on=['ano', 'id_municipio', 'rede_descricao'],
    right_on=['ano', 'id_municipio', 'rede_descricao'],
    how='inner',
    suffixes=('_oficial', '_meta')
)

total_linhas_resultantes_municipio = len(df_gold_municipio)

# VALIDAÇÃO DA TAXA DE CORRESPONDÊNCIA (Exigência do edital)
taxa_match = (total_linhas_resultantes_municipio / total_linhas_originais_municipio) * 100
LIMIAR_MINIMO = 40.0

print(f"\n[AUDITORIA DE JOIN - MUNICIPIO] Total antes do join: {total_linhas_originais_municipio}")
print(f"[AUDITORIA DE JOIN - MUNICIPIO] Total após o inner join: {total_linhas_resultantes_municipio}")
print(f"[AUDITORIA DE JOIN - MUNICIPIO] Taxa de correspondência (Match Rate): {taxa_match:.2f}%")

sucesso_processamento_municipio = True

if taxa_match < LIMIAR_MINIMO:
    print(f"{Fore.RED}[ERRO CRÍTICO DE QUALIDADE] Taxa de match ({taxa_match:.2f}%) ficou abaixo do limiar de {LIMIAR_MINIMO}%!")
    print(f"{Fore.YELLOW}[CONTINGÊNCIA] A camada Gold para 'municipio_metas' NÃO será gerada para evitar propagação de dados corrompidos.")
    sucesso_processamento_municipio = False
else:
    print(f"{Fore.GREEN}[QUALIDADE APROVADA] Taxa de correspondência dentro do limiar aceitável.")

# 4 e 5. Limpeza e salvamento condicional (Apenas se passou na qualidade)
if sucesso_processamento_municipio:
    df_gold_municipio = df_gold_municipio.drop(columns=['rede_descricao'])
    gold_dfs['municipio_metas'] = df_gold_municipio
    output_file_gold = f"{GOLD_PATH}/municipio_metas_gold.parquet"
    df_gold_municipio.to_parquet(output_file_gold, index=False)

    print(f"{Fore.GREEN}Entidade MUNICIPIO_METAS processada com sucesso na camada Gold!")
    print(f"Salvo em: {output_file_gold}")
    print(f"Total de registros após o Inner Join: {len(df_gold_municipio)}")
    print(f"Colunas resultantes: {list(df_gold_municipio.columns)}\n")
else:
    print(f"{Fore.MAGENTA}[PULANDO ETAPA] Execução prosseguindo normalmente para as próximas tabelas\n")

Processando consolidação Gold: Município e Metas

[AUDITORIA] Todos os códigos de 'rede' foram mapeados com sucesso pelo dicionário.

[AUDITORIA DE JOIN - MUNICIPIO] Total antes do join: 23995
[AUDITORIA DE JOIN - MUNICIPIO] Total após o inner join: 10654
[AUDITORIA DE JOIN - MUNICIPIO] Taxa de correspondência (Match Rate): 44.40%
[QUALIDADE APROVADA] Taxa de correspondência dentro do limiar aceitável.
Entidade MUNICIPIO_METAS processada com sucesso na camada Gold!
Salvo em: ./data_lake/gold/municipio_metas_gold.parquet
Total de registros após o Inner Join: 10654
Colunas resultantes: ['ano', 'id_municipio', 'serie', 'rede_oficial', 'taxa_alfabetizacao_oficial', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ingestion_timestamp_oficial', '_source_table_oficial', '_source_sy

In [72]:
# ============================================================
# ETAPA 4.2 — CONSOLIDAÇÃO UF E METAS
# ============================================================
print(f"{Fore.YELLOW}Processando consolidação Gold: UF e Metas\n")

# 1. Carregar os dados limpos da Silver
df_uf_silver = pd.read_parquet(f"{SILVER_PATH}/uf_silver.parquet")
df_meta_uf_silver = pd.read_parquet(f"{SILVER_PATH}/meta_alfabetizacao_uf_silver.parquet")

# 2. Criar o mapeamento usando a tabela de dicionário
df_map_rede_uf = df_dicionario_silver[
    (df_dicionario_silver['id_tabela'] == 'uf') & 
    (df_dicionario_silver['nome_coluna'] == 'rede')
][['chave', 'valor']].set_index('chave')['valor'].to_dict()

# 3. Traduzir os códigos numéricos da UF para as descrições oficiais do dicionário
df_uf_silver['rede_descricao'] = df_uf_silver['rede'].map(df_map_rede_uf)

# CORREÇÃO E AUDITORIA: Verifica e preenche códigos fora do dicionário com 'Desconhecido'
nulos_pos_map_uf = df_uf_silver['rede_descricao'].isnull().sum()
if nulos_pos_map_uf > 0:
    print(f"{Fore.YELLOW}[ALERTA DE AUDITORIA] Encontrados {nulos_pos_map_uf} registros com códigos fora do dicionário na tabela UF ('rede'). Atribuindo 'Desconhecido'.")
    df_uf_silver['rede_descricao'] = df_uf_silver['rede_descricao'].fillna('Desconhecido')
else:
    print(f"{Fore.GREEN}[AUDITORIA] Todos os códigos de 'rede' da tabela UF foram mapeados com sucesso.")

# 4. Padronização semântica robusta: 
# Se a descrição contiver a palavra "Pública", unificamos para "Pública" 
# para alinhar perfeitamente com a granularidade da tabela de metas, que tem sua descrição 
# diferente de como é na tabela de dicionário
df_uf_silver['rede_descricao'] = df_uf_silver['rede_descricao'].apply(
    lambda x: 'Pública' if isinstance(x, str) and 'Pública' in x else x
)

# Na tabela de metas, a coluna de rede já vem como 'Pública'
df_meta_uf_silver['rede_descricao'] = df_meta_uf_silver['rede']

# 5. Realizando o inner join utilizando as descrições padronizadas
total_linhas_originais_uf = len(df_uf_silver)

df_gold_uf = pd.merge(
    df_uf_silver,
    df_meta_uf_silver,
    left_on=['ano', 'sigla_uf', 'rede_descricao'],
    right_on=['ano', 'sigla_uf', 'rede_descricao'],
    how='inner',
    suffixes=('_oficial', '_meta')
)

total_linhas_resultantes_uf = len(df_gold_uf)

# VALIDAÇÃO DA TAXA DE CORRESPONDÊNCIA PARA UF (Exigência do edital)
taxa_match_uf = (total_linhas_resultantes_uf / total_linhas_originais_uf) * 100
LIMIAR_MINIMO_UF = 30.0

print(f"\n[AUDITORIA DE JOIN - UF] Total antes do join: {total_linhas_originais_uf}")
print(f"[AUDITORIA DE JOIN - UF] Total após o inner join: {total_linhas_resultantes_uf}")
print(f"[AUDITORIA DE JOIN - UF] Taxa de correspondência (Match Rate): {taxa_match_uf:.2f}%")

sucesso_processamento_uf = True

if taxa_match_uf < LIMIAR_MINIMO_UF:
    print(f"{Fore.RED}[ERRO CRÍTICO DE QUALIDADE] Taxa de match da UF ({taxa_match_uf:.2f}%) ficou abaixo do limiar de {LIMIAR_MINIMO_UF}%!")
    print(f"{Fore.YELLOW}[CONTINGÊNCIA] A camada Gold para 'uf_metas' NÃO será gerada para evitar propagação de dados corrompidos.")
    sucesso_processamento_uf = False
else:
    print(f"{Fore.GREEN}[QUALIDADE APROVADA] Taxa de correspondência da UF dentro do limiar aceitável.")

# 6 e 7. Limpeza e salvamento condicional (Apenas se passou na qualidade)
if sucesso_processamento_uf:
    df_gold_uf = df_gold_uf.drop(columns=['rede_descricao'])
    gold_dfs['uf_metas'] = df_gold_uf
    output_file_gold_uf = f"{GOLD_PATH}/uf_metas_gold.parquet"
    df_gold_uf.to_parquet(output_file_gold_uf, index=False)

    print(f"{Fore.GREEN}Entidade UF_METAS processada com sucesso na camada Gold!")
    print(f"Salvo em: {output_file_gold_uf}")
    print(f"Total de registros após o Inner Join: {len(df_gold_uf)}")
    print(f"Colunas resultantes: {list(df_gold_uf.columns)}\n")
else:
    print(f"{Fore.MAGENTA}[PULANDO ETAPA] Execução prosseguindo normalmente para as próximas tabelas\n")

Processando consolidação Gold: UF e Metas

[AUDITORIA] Todos os códigos de 'rede' da tabela UF foram mapeados com sucesso.

[AUDITORIA DE JOIN - UF] Total antes do join: 145
[AUDITORIA DE JOIN - UF] Total após o inner join: 49
[AUDITORIA DE JOIN - UF] Taxa de correspondência (Match Rate): 33.79%
[QUALIDADE APROVADA] Taxa de correspondência da UF dentro do limiar aceitável.
Entidade UF_METAS processada com sucesso na camada Gold!
Salvo em: ./data_lake/gold/uf_metas_gold.parquet
Total de registros após o Inner Join: 49
Colunas resultantes: ['ano', 'sigla_uf', 'serie', 'rede_oficial', 'taxa_alfabetizacao_oficial', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8', '_ingestion_timestamp_oficial', '_source_table_oficial', '_source_system_oficial', '_record_hash_oficial', 'rede_meta',

In [73]:
# ============================================================
# ETAPA 4.3 — ENRIQUECIMENTO DINÂMICO DA TABELA ALUNOS
# ============================================================
print(f"{Fore.YELLOW}Processando consolidação Gold: Alunos\n")

# 1. Carregar os dados limpos da Silver
df_alunos_silver = pd.read_parquet(f"{SILVER_PATH}/alunos_silver.parquet")

# 2. Descobrir dinamicamente quais colunas da tabela 'alunos' possuem mapeamento no dicionário
colunas_para_mapear = df_dicionario_silver[df_dicionario_silver['id_tabela'] == 'alunos']['nome_coluna'].unique()

print(f"Colunas identificadas dinamicamente para tradução: {list(colunas_para_mapear)}")

# 3. Função de mapeamento genérica e dinâmica
df_gold_alunos = df_alunos_silver.copy()

for coluna in colunas_para_mapear:
    # Extrai o dicionário (chave -> valor) para a coluna atual
    mapa = df_dicionario_silver[
        (df_dicionario_silver['id_tabela'] == 'alunos') & 
        (df_dicionario_silver['nome_coluna'] == coluna)
    ][['chave', 'valor']].set_index('chave')['valor'].to_dict()
    
    if mapa and coluna in df_gold_alunos.columns:
        coluna_destino = f"{coluna}_descricao"
        df_gold_alunos[coluna_destino] = df_gold_alunos[coluna].map(mapa)

        # AUDITORIA: Verifica se algum código ficou sem tradução (virou nulo)
        nulos_mapa = df_gold_alunos[coluna_destino].isnull().sum()
        total_preenchidos = df_gold_alunos[coluna].notnull().sum()
        
        if nulos_mapa > 0 and total_preenchidos > 0:
            print(f"{Fore.YELLOW}[AUDITORIA] Coluna '{coluna}': {nulos_mapa} registros com códigos fora do dicionário. Atribuindo 'Desconhecido'.")
            df_gold_alunos[coluna_destino] = df_gold_alunos[coluna_destino].fillna('Desconhecido')
        else:
            print(f"{Fore.GREEN}[AUDITORIA] Coluna '{coluna}': Todos os códigos mapeados com sucesso.")

# 4. Salvando na camada Gold
gold_dfs['alunos'] = df_gold_alunos
output_file_gold_alunos = f"{GOLD_PATH}/alunos_gold.parquet"
df_gold_alunos.to_parquet(output_file_gold_alunos, index=False)

print(f"{Fore.GREEN}Entidade ALUNOS processada com sucesso na camada Gold!")
print(f"Salvo em: {output_file_gold_alunos}")
print(f"Total de registros: {len(df_gold_alunos)}")
print(f"Colunas resultantes: {list(df_gold_alunos.columns)}\n")

Processando consolidação Gold: Alunos

Colunas identificadas dinamicamente para tradução: ['alfabetizado', 'preenchimento_caderno', 'presenca', 'rede', 'serie']
[AUDITORIA] Coluna 'alfabetizado': Todos os códigos mapeados com sucesso.
[AUDITORIA] Coluna 'preenchimento_caderno': Todos os códigos mapeados com sucesso.
[AUDITORIA] Coluna 'presenca': Todos os códigos mapeados com sucesso.
[AUDITORIA] Coluna 'rede': Todos os códigos mapeados com sucesso.
[AUDITORIA] Coluna 'serie': Todos os códigos mapeados com sucesso.
Entidade ALUNOS processada com sucesso na camada Gold!
Salvo em: ./data_lake/gold/alunos_gold.parquet
Total de registros: 3867999
Colunas resultantes: ['ano', 'id_municipio', 'id_escola', 'id_aluno', 'caderno', 'serie', 'rede', 'presenca', 'preenchimento_caderno', 'alfabetizado', 'proficiencia', 'peso_aluno', '_ingestion_timestamp', '_source_table', '_source_system', '_record_hash', 'alfabetizado_descricao', 'preenchimento_caderno_descricao', 'presenca_descricao', 'rede_desc